In [ ]:
!git clone https://github.com/UefiMemAnalysis/UefiMemAnalysis.git

Cloning into 'UefiMemAnalysis'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 129 (delta 54), reused 95 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 105.89 KiB | 2.16 MiB/s, done.
Resolving deltas: 100% (54/54), done.


In [ ]:
%cd UefiMemAnalysis

/content/UefiMemAnalysis


In [ ]:
!ls -la


total 32
drwxr-xr-x 5 root root 4096 Aug  3 11:53 .
drwxr-xr-x 1 root root 4096 Aug  3 11:53 ..
drwxr-xr-x 8 root root 4096 Aug  3 11:53 .git
-rw-r--r-- 1 root root  663 Aug  3 11:53 .gitignore
-rw-r--r-- 1 root root 1072 Aug  3 11:53 LICENSE
-rw-r--r-- 1 root root 3540 Aug  3 11:53 README.md
drwxr-xr-x 3 root root 4096 Aug  3 11:53 UEFIDumpAnalysis
drwxr-xr-x 4 root root 4096 Aug  3 11:53 UefiMemDump


In [ ]:
!cat README.md

# UefiMemAnalysis

`UefiMemAnalysis` is an open-source framework for UEFI memory acquisition and
offline analysis of UEFI memory dumps. It accompanies the paper
[UEFI Memory Forensics](https://arxiv.org/pdf/2501.16962), which has been
accepted to the 11th IEEE European Symposium on Security and Privacy
(Euro S&P).

The repository includes acquisition tooling for collecting UEFI memory dumps and
an analysis toolkit for investigating UEFI memory artifacts and detecting
suspicious runtime behavior.

## Repository Layout

- `UefiMemDump/`
  Acquisition component providing:
  - an EDK II DXE driver intended for integration into firmware images; it writes the dump to removable media during the firmware-to-OS handoff
  - a UEFI shell application that performs the same dump-to-removable-media workflow without requiring firmware integration
  - a utility, `concat_dump_files.py`, that reassembles split dump chunks into a single dump file
- `UEFIDumpAnalysis/`
  Analysis component with plugins fo

In [ ]:
!python -m pip install -e ./UEFIDumpAnalysis

Obtaining file:///content/UefiMemAnalysis/UEFIDumpAnalysis
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 20.5 MB/s eta 0:00:00
  Building editable for uefi-dump-analysis (pyproject.toml) ... done
  Created wheel for uefi-dump-analysis: filename=uefi_dump_analysis-0.1.0-0.editable-py3-none-any.whl size=7116 sha256=ac96208a6e96866024c9d510c989b34bce93a40a4ee8100d63456041c6638f26
  Stored in directory: /tmp/pip-ephem-wheel-cache-bc5_i5yk/wheels/21/bc/bb/ec11b1921a566a7a4d59e485517d6ba595fa01efe3067c9afe
Successfully built uefi-dump-analysis


In [ ]:
!python -m uefi_dump_analysis -h

usage: __main__.py [-h]
                   {gadget_detection,image_load_path_detection,inline_hooking_detection,pointer_hooking_detection,uefi_image_carving}
                   ...

Plugin-based CLI for analyzing UEFI memory dumps.
Run 'uefi-mem-analysis <module> -h', or 'python -m uefi_dump_analysis <module> -h' for module-specific help.

options:
  -h, --help            show this help message and exit

modules:
  {gadget_detection,image_load_path_detection,inline_hooking_detection,pointer_hooking_detection,uefi_image_carving}
                        Available analysis modules
    gadget_detection    Resolve candidate gadget addresses using Ropper output. Accepts either carved analysis images or a dump that will be carved automatically before analysis.
    image_load_path_detection
                        Identify path-backed loaded images whose exact paths are not present in the configured whitelist.
    inline_hooking_detection
                        Analyze EFI Boot, Runtime, and 

In [ ]:
!cat UefiMemDump/docs/qemu-testing.md

# QEMU Testing

This document covers the QEMU workflow for implementation and testing of
`UefiMemDump`, especially the DXE driver path.

## Key Driver Behavior

`UefiMemDumpDriver` is designed to dump memory near `ExitBootServices`. To
simulate a real system as closely as possible, the recommended QEMU setup boots
an actual guest OS from a virtual disk.

Testing the driver on a plain firmware-only boot is not representative and can
fail to exercise the intended path, because the dump is tied to the
pre-`ExitBootServices` event. In practice, booting a Windows guest is the
recommended test setup.

For preparing that Windows guest disk, see:

- [windows-vhd-manual.md](windows-vhd-manual.md)

## Create A Virtual Removable-Media Image

If you want the guest to dump to removable media in QEMU, create a removable-
media image. A 4 GB image is usually enough for a 1 GB guest memory
configuration, but size it according to the memory you configured in QEMU.

The example below was used on Ubuntu.

In [ ]:
!find . -iname "*.bin" -o -iname "*dump*" | grep -v ".git"


./UEFIDumpAnalysis
./UEFIDumpAnalysis/uefi_dump_analysis
./UEFIDumpAnalysis/uefi_dump_analysis.egg-info
./UefiMemDump
./UefiMemDump/concat_dump_files.py
./UefiMemDump/edk2/UefiMemDumpApp
./UefiMemDump/edk2/UefiMemDumpApp/UefiMemDump.c
./UefiMemDump/edk2/UefiMemDumpApp/UefiMemDump.h
./UefiMemDump/edk2/UefiMemDumpApp/UefiMemDump.inf
./UefiMemDump/edk2/UefiMemDumpDriver
./UefiMemDump/edk2/UefiMemDumpDriver/UefiMemDump.c
./UefiMemDump/edk2/UefiMemDumpDriver/UefiMemDump.h
./UefiMemDump/edk2/UefiMemDumpDriver/UefiMemDump.inf


In [ ]:
!find . -iname "test*" -o -iname "sample*" | grep -v ".git"

In [ ]:
!ls -la UEFIDumpAnalysis/

total 40
drwxr-xr-x 4 root root  4096 Aug  3 11:55 .
drwxr-xr-x 5 root root  4096 Aug  3 11:53 ..
-rw-r--r-- 1 root root   514 Aug  3 11:53 pyproject.toml
-rw-r--r-- 1 root root 13482 Aug  3 11:53 README.md
-rw-r--r-- 1 root root    18 Aug  3 11:53 requirements.txt
drwxr-xr-x 6 root root  4096 Aug  3 11:55 uefi_dump_analysis
drwxr-xr-x 2 root root  4096 Aug  3 11:55 uefi_dump_analysis.egg-info


In [ ]:
!find UEFIDumpAnalysis/uefi_dump_analysis -type f -name "*.py" | head -30

UEFIDumpAnalysis/uefi_dump_analysis/__main__.py
UEFIDumpAnalysis/uefi_dump_analysis/utilities/memory_utils.py
UEFIDumpAnalysis/uefi_dump_analysis/utilities/constants.py
UEFIDumpAnalysis/uefi_dump_analysis/utilities/__init__.py
UEFIDumpAnalysis/uefi_dump_analysis/utilities/parsing_utils.py
UEFIDumpAnalysis/uefi_dump_analysis/__init__.py
UEFIDumpAnalysis/uefi_dump_analysis/modules/uefi_image_carving.py
UEFIDumpAnalysis/uefi_dump_analysis/modules/pointer_hooking_detection.py
UEFIDumpAnalysis/uefi_dump_analysis/modules/inline_hooking_detection.py
UEFIDumpAnalysis/uefi_dump_analysis/modules/__init__.py
UEFIDumpAnalysis/uefi_dump_analysis/modules/gadget_detection.py
UEFIDumpAnalysis/uefi_dump_analysis/modules/image_load_path_detection.py
UEFIDumpAnalysis/uefi_dump_analysis/cli/__main__.py
UEFIDumpAnalysis/uefi_dump_analysis/cli/__init__.py


In [ ]:
!find . -iname "test*" -type d


In [ ]:
!find . -iname "test*" -type d

In [ ]:
!cat UEFIDumpAnalysis/uefi_dump_analysis/modules/uefi_image_carving.py

"""Carve loaded UEFI images and emit the associated metadata from memory dumps."""

import csv
import json
import os
import re
import struct
from dataclasses import dataclass
from typing import Any, Optional

from uefi_dump_analysis.utilities import constants as cs
from uefi_dump_analysis.utilities import memory_utils as mu

MIN_LOADED_IMAGE_READ_SIZE = cs.IMAGE_SIZE_OFFSET + 8


@dataclass(frozen=True)
class ImageRecord:
    struct_offset: int
    revision: int
    system_table_pointer: int
    system_table_signature_valid: bool
    image_base: int
    image_size: int
    image_end: int
    file_path_pointer: int
    identity: str = "unknown"


@dataclass(frozen=True)
class PeHeaderCandidate:
    offset: int
    image_base: int
    size_of_image: int


def _classify_loaded_image_candidate(data, signature_offset):
    """Validate one signature hit and return either a record or a rejection reason."""
    if signature_offset + MIN_LOADED_IMAGE_READ_SIZE > len(data):
        return None, 

In [ ]:
!cat UEFIDumpAnalysis/uefi_dump_analysis/utilities/constants.py

# Constants for Boot, Runtime, and DXE Services
EFI_BOOT_SERVICES_SIGNATURE = b'BOOTSERV'
EFI_RUNTIME_SERVICES_SIGNATURE = b'RUNTSERV'
EFI_DXE_SERVICES_SIGNATURE = b'DXE_SERV'

EFI_BOOT_SERVICES_SIZE = 376
EFI_RUNTIME_SERVICES_SIZE = 136
EFI_DXE_SERVICES_SIZE = 168

# Constants for image extraction
SIGNATURE = b'ldri'
IMAGE_REVISION_OFFSET = 40  # EFI_LOADED_IMAGE_PROTOCOL.Revision
SYSTEM_TABLE_OFFSET = 56  # EFI_LOADED_IMAGE_PROTOCOL.SystemTable
IMAGE_BASE_OFFSET = 104  # Offset of ImageBasePage within the structure
IMAGE_SIZE_OFFSET = 112  # Offset of ImageSize within EFI_LOADED_IMAGE_PROTOCOL inside the structure
GUID_OFFSET = 72  # Offset from the beginning of the signature to the pointer address
GUID_SIZE = 16  # Size of the GUID in bytes
EFI_LOADED_IMAGE_PROTOCOL_REVISION = 0x1000
EFI_SYSTEM_TABLE_SIGNATURE = 0x5453595320494249  # 'IBI SYST' in little-endian

TABLES_HEADER_SIZE = 24

# Device path constants
MEDIA_DEVICE_PATH = 0x04
MEDIA_FILEPATH_DP = 0x04
MEDIA_FW_VOL_FILEPATH_D

In [ ]:
!cat UEFIDumpAnalysis/uefi_dump_analysis/utilities/memory_utils.py

import mmap
import os
import re
import struct
from collections import Counter
from bisect import bisect_right
from dataclasses import dataclass
from typing import Optional

from uefi_dump_analysis.utilities import constants as cs


@dataclass(frozen=True)
class AddressRegion:
    start: int
    end: int
    file_offset_start: int


class AddressTranslator:
    """
    Runtime-address to dump-file-offset translator.

    If no regions are provided, fallback mode assumes identity mapping
    (address == file offset), which only works for some dump layouts.
    """

    def __init__(self, dump_size, regions=None):
        self.dump_size = dump_size
        self.regions = regions or []
        self._region_starts = [r.start for r in self.regions]

    def _region_index_for_address(self, address):
        if not self.regions:
            return None
        index = bisect_right(self._region_starts, address) - 1
        if index < 0:
            return None
        region = self.regions[inde

In [ ]:
import struct

# 256 byte'lık sıfırlarla dolu bir "sahte bellek dökümü" oluştur
buf = bytearray(256)

SIG_OFFSET = 0
buf[SIG_OFFSET:SIG_OFFSET+4] = b'ldri'                                   # SIGNATURE

# Revision (offset 40, 4 byte) -> EFI_LOADED_IMAGE_PROTOCOL_REVISION
struct.pack_into("<I", buf, SIG_OFFSET + 40, 0x1000)

# SystemTable pointer (offset 56, 8 byte) -> dump sınırları içinde bir adres olsun (identity-map varsayımıyla)
struct.pack_into("<Q", buf, SIG_OFFSET + 56, 200)

# GUID/file path pointer (offset 72, 8 byte) -> kasıtlı olarak dump dışı, identity "unknown" dönecek
struct.pack_into("<Q", buf, SIG_OFFSET + 72, 0xFFFFFFFFFFFFFFFF)

# ImageBasePage (offset 104, 8 byte)
struct.pack_into("<Q", buf, SIG_OFFSET + 104, 0x400000)

# ImageSize (offset 112, 8 byte)
struct.pack_into("<Q", buf, SIG_OFFSET + 112, 0x1000)

with open('/content/fake_uefi_dump.bin', 'wb') as f:
    f.write(buf)

print("Sahte dump oluşturuldu:", len(buf), "byte")

Sahte dump oluşturuldu: 256 byte


In [ ]:
!python -m uefi_dump_analysis uefi_image_carving -f /content/fake_uefi_dump.bin -o /content/uefi_results -debug

Detected 1 unique loaded image records
Metadata CSV: /content/uefi_results/images.csv
Metadata JSON: /content/uefi_results/images.json
[debug] UEFIImageCarving diagnostics
[debug] Translation mode: identity-fallback
[debug] Signature hits: 1
[debug] Candidate rejection counts:
[debug]   too_short: 0
[debug]   bad_revision: 0
[debug]   null_system_table_pointer: 0
[debug]   zero_base_or_size: 0
[debug]   oversized_image: 0
[debug]   wrapped_image_range: 0
[debug] Candidate summary: accepted=1 system_table_signature_unreadable=0 signature_valid=0
[debug] Filtering summary: dominant_system_table=0x00000000000000C8 require_signature_valid=False filtered_out_other_system_table=0 filtered_out_invalid_signature=0 retained=1
[debug] Deduplication summary: replaced_records=0 unique_records=1
